In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip install rdflib

In [ ]:
import urllib.request
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import os

# 1. Download Google's Noto Sans Sinhala directly to Kaggle working directory
font_url = "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansSinhala/NotoSansSinhala-Regular.ttf"
font_path = "/kaggle/working/NotoSansSinhala.ttf"

if not os.path.exists(font_path):
    print("Downloading Sinhala Font...")
    urllib.request.urlretrieve(font_url, font_path)
    print("Download complete.")

# 2. Register the font with Matplotlib
font_manager.fontManager.addfont(font_path)
sinhala_font = font_manager.FontProperties(fname=font_path)

# 3. Force Matplotlib to use this font globally for all text
plt.rcParams['font.family'] = sinhala_font.get_name()

print(f"Success! Matplotlib is now using: {plt.rcParams['font.family']}")

In [ ]:
from rdflib import Graph, Literal, Namespace, RDF, RDFS, URIRef
import os

def build_sinhala_kg():
    # 1. Initialize the Graph and Namespace
    g = Graph()
    sin = Namespace("http://research.lk/ontology/sinhala#")
    g.bind("sin", sin)  # Binds the prefix 'sin:' to our namespace for readability

    print("Building ontology classes and properties...")

    # ==========================================
    # 2. DEFINE CLASSES (Ontology Hierarchy)
    # ==========================================
    g.add((sin.Character, RDF.type, RDFS.Class))
    g.add((sin.BaseConsonant, RDFS.subClassOf, sin.Character))
    g.add((sin.Modifier, RDFS.subClassOf, sin.Character))
    g.add((sin.Syllable, RDFS.subClassOf, sin.Character))

    # ==========================================
    # 3. DEFINE PROPERTIES (Edges/Relationships)
    # ==========================================
    g.add((sin.hasBase, RDF.type, RDF.Property))
    g.add((sin.modifiedBy, RDF.type, RDF.Property))
    g.add((sin.romanizedAs, RDF.type, RDF.Property))
    
    # Restrict domains and ranges for clean ontology rules
    g.add((sin.hasBase, RDFS.domain, sin.Syllable))
    g.add((sin.hasBase, RDFS.range, sin.BaseConsonant))
    g.add((sin.modifiedBy, RDFS.domain, sin.Syllable))
    g.add((sin.modifiedBy, RDFS.range, sin.Modifier))

    # ==========================================
    # 4. RAW DATA DICTIONARIES (Complete Set)
    # ==========================================
    
    # All 42 Base Consonants in the Sinhala Alphabet
    consonants = {
        # Kanthya (Gutturals)
        "Ka": {"char": "ක", "roman_base": "k"},
        "Kha": {"char": "ඛ", "roman_base": "kh"},
        "Ga": {"char": "ග", "roman_base": "g"},
        "Gha": {"char": "ඝ", "roman_base": "gh"},
        "Nga": {"char": "ඞ", "roman_base": "ng"},
        "Sanyaka_Ga": {"char": "ඟ", "roman_base": "nng"},

        # Thaluja (Palatals)
        "Ca": {"char": "ච", "roman_base": "c"},
        "Cha": {"char": "ඡ", "roman_base": "ch"},
        "Ja": {"char": "ජ", "roman_base": "j"},
        "Jha": {"char": "ඣ", "roman_base": "jh"},
        "Nya": {"char": "ඤ", "roman_base": "ny"},
        "Jnya": {"char": "ඥ", "roman_base": "jny"},
        "Sanyaka_Ja": {"char": "ඦ", "roman_base": "nnj"},

        # Moordhaja (Retroflexes)
        "Ta": {"char": "ට", "roman_base": "t"},
        "Tha": {"char": "ඨ", "roman_base": "th"},
        "Da": {"char": "ඩ", "roman_base": "d"},
        "Dha": {"char": "ඪ", "roman_base": "dh"},
        "Nna": {"char": "ණ", "roman_base": "nn"},
        "Sanyaka_Da": {"char": "ඬ", "roman_base": "nnd"},

        # Danthya (Dentals)
        "Ta_Dental": {"char": "ත", "roman_base": "th"},
        "Tha_Dental": {"char": "ථ", "roman_base": "thh"},
        "Da_Dental": {"char": "ද", "roman_base": "d"},
        "Dha_Dental": {"char": "ධ", "roman_base": "dh"},
        "Na": {"char": "න", "roman_base": "n"},
        "Sanyaka_Da_Dental": {"char": "ඳ", "roman_base": "nd"},

        # Oshtya (Labials)
        "Pa": {"char": "ප", "roman_base": "p"},
        "Pha": {"char": "ඵ", "roman_base": "ph"},
        "Ba": {"char": "බ", "roman_base": "b"},
        "Bha": {"char": "භ", "roman_base": "bh"},
        "Ma": {"char": "ම", "roman_base": "m"},
        "Sanyaka_Ba": {"char": "ඹ", "roman_base": "mb"},

        # Approximants, Fricatives, and Others
        "Ya": {"char": "ය", "roman_base": "y"},
        "Ra": {"char": "ර", "roman_base": "r"},
        "La": {"char": "ල", "roman_base": "l"},
        "Va": {"char": "ව", "roman_base": "v"},
        "Sha": {"char": "ශ", "roman_base": "sh"},
        "Shha": {"char": "ෂ", "roman_base": "shh"},
        "Sa": {"char": "ස", "roman_base": "s"},
        "Ha": {"char": "හ", "roman_base": "h"},
        "Lla": {"char": "ළ", "roman_base": "ll"},
        "Fa": {"char": "ෆ", "roman_base": "f"}
    }

    # All 16 Standard Modifiers (පිල්ල - Pilla)
    modifiers = {
        "HalKireema": {"char": "්", "roman_suffix": ""},        # Hal kireema (Removes inherent 'a')
        "AelaPilla": {"char": "ා", "roman_suffix": "aa"},       # Aela pilla (Long a)
        "KetiAedaPilla": {"char": "ැ", "roman_suffix": "ae"},   # Keti aeda pilla (Short ae)
        "DigaAedaPilla": {"char": "ෑ", "roman_suffix": "aae"},  # Diga aeda pilla (Long ae)
        "KetiIspilla": {"char": "ි", "roman_suffix": "i"},      # Keti ispilla (Short i)
        "DigaIspilla": {"char": "ී", "roman_suffix": "ii"},     # Diga ispilla (Long i)
        "KetiPaapilla": {"char": "ු", "roman_suffix": "u"},      # Keti paapilla (Short u)
        "DigaPaapilla": {"char": "ූ", "roman_suffix": "uu"},     # Diga paapilla (Long u)
        "GaettaPilla": {"char": "ෘ", "roman_suffix": "ru"},     # Gaetta pilla (Vocalic r)
        "DigaGaettaPilla": {"char": "ෟ", "roman_suffix": "ruu"},# Diga gaetta pilla (Long vocalic r)
        "Kombuwa": {"char": "ෙ", "roman_suffix": "e"},        # Kombuwa (Short e)
        "DigaKombuwa": {"char": "ේ", "roman_suffix": "ee"},     # Diga kombuwa (Long e)
        "KombuDeka": {"char": "ෛ", "roman_suffix": "ai"},       # Kombu deka (ai)
        "KombuwaSahaAelaPilla": {"char": "ො", "roman_suffix": "o"}, # Kombuwa + Aela pilla (Short o)
        "KombuwaSahaDigaAelaPilla": {"char": "ෝ", "roman_suffix": "oo"}, # Kombuwa + Diga aela pilla (Long o)
        "KombuwaSahaHalKireema": {"char": "ෞ", "roman_suffix": "au"}  # Kombuwa + Hal kireema (au)
    }

    print("Injecting Base Consonants and Modifiers into the graph...")

    # ==========================================
    # 5. POPULATE BASE NODES
    # ==========================================
    for key, data in consonants.items():
        node = sin[key]
        g.add((node, RDF.type, sin.BaseConsonant))
        g.add((node, RDFS.label, Literal(data["char"], lang="si")))
        # Note: The base character has an inherent 'a' sound (e.g., ක = ka)
        g.add((node, sin.romanizedAs, Literal(data["roman_base"] + "a"))) 

    for key, data in modifiers.items():
        node = sin[key]
        g.add((node, RDF.type, sin.Modifier))
        g.add((node, RDFS.label, Literal(data["char"], lang="si")))

    print("Programmatically generating Syllables...")

    # ==========================================
    # 6. PROGRAMMATICALLY GENERATE SYLLABLES
    # ==========================================
    # This loop generates every combination of Consonant + Modifier
    for c_key, c_data in consonants.items():
        for m_key, m_data in modifiers.items():
            
            # Create a unique URI key (e.g., 'Ka_AelaPilla')
            s_key = f"{c_key}_{m_key}"
            
            # Combine the Unicode characters exactly how Sinhala typing works
            s_char = c_data["char"] + m_data["char"]
            
            # Combine the romanized text
            s_roman = c_data["roman_base"] + m_data["roman_suffix"]

            # Add to graph
            s_node = sin[s_key]
            g.add((s_node, RDF.type, sin.Syllable))
            g.add((s_node, RDFS.label, Literal(s_char, lang="si")))
            g.add((s_node, sin.hasBase, sin[c_key]))
            g.add((s_node, sin.modifiedBy, sin[m_key]))
            g.add((s_node, sin.romanizedAs, Literal(s_roman)))

    # ==========================================
    # 7. SERIALIZE TO FILE
    # ==========================================
    output_file = "sinhala_kg.ttl"
    print(f"Serializing graph to {output_file}...")
    
    # Save the graph in Turtle format
    g.serialize(destination=output_file, format="turtle")
    
    print(f"Success! Graph created with {len(g)} triples.")

if __name__ == "__main__":
    build_sinhala_kg()

In [ ]:
import rdflib
import re

def tokenize_sinhala_string(text):
    """
    Groups Sinhala base consonants with their attached modifiers (පිල්ල).
    Sinhala modifiers generally fall in the Unicode range \u0DCA to \u0DDF.
    """
    tokens = []
    for char in text:
        # Check if the character is a modifier (including Hal Kireema)
        # \u0D82 is Anuswaraya (ං), \u0D83 is Visargaya (ඃ)
        is_modifier = ('\u0DCA' <= char <= '\u0DDF') or char in ['\u0D82', '\u0D83']
        
        if is_modifier:
            if tokens:
                tokens[-1] += char # Attach to the previous consonant
            else:
                tokens.append(char) # Fallback if text starts weirdly
        elif char.strip(): # Ignore whitespace
            tokens.append(char)
            
    return tokens

def extract_subgraph(input_text, kg_path="sinhala_kg.ttl"):
    # 1. Load the foundational Knowledge Graph
    print(f"Loading main KG from {kg_path}...")
    main_graph = rdflib.Graph()
    main_graph.parse(kg_path, format="turtle")
    
    # 2. Segment the input text
    tokens = tokenize_sinhala_string(input_text)
    print(f"Segmented Input: {tokens}")
    
    # 3. Initialize the empty Subgraph
    subgraph = rdflib.Graph()
    sin = rdflib.Namespace("http://research.lk/ontology/sinhala#")
    subgraph.bind("sin", sin)
    
    # 4. Extract data using SPARQL CONSTRUCT
    # This query finds the token, grabs its properties, and crucially, 
    # reaches out to grab the properties of its Base and Modifier as well.
    for token in tokens:
        query = f"""
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX sin: <http://research.lk/ontology/sinhala#>
        
        CONSTRUCT {{
            ?node ?p ?o .
            ?base ?base_p ?base_o .
            ?mod ?mod_p ?mod_o .
        }}
        WHERE {{
            # 1. Find the node matching the exact syllable label
            ?node rdfs:label "{token}"@si .
            ?node ?p ?o .
            
            # 2. Optionally, fetch all data about its Base Consonant
            OPTIONAL {{
                ?node sin:hasBase ?base .
                ?base ?base_p ?base_o .
            }}
            
            # 3. Optionally, fetch all data about its Modifier
            OPTIONAL {{
                ?node sin:modifiedBy ?mod .
                ?mod ?mod_p ?mod_o .
            }}
        }}
        """
        # Execute query and merge results into our subgraph
        result_graph = main_graph.query(query)
        for triple in result_graph:
            subgraph.add(triple)

    return subgraph

if __name__ == "__main__":
    # Test with a Sinhala word (e.g., "කාලය" - Time)
    input_word = "සාදරයෙන් පිළිගනිමු" 
    print(f"Processing input: {input_word}")
    
    # Generate the subgraph
    specific_kg = extract_subgraph(input_word, "sinhala_kg.ttl")
    
    print("\n--- Extracted Subgraph ---")
    print(specific_kg.serialize(format="turtle"))
    print(f"Total triples extracted: {len(specific_kg)}")

In [ ]:
!pip install rdflib pyvis networkx

In [ ]:
import rdflib
from pyvis.network import Network
import os
import webbrowser

def visualize_rdf_graph(ttl_file, output_html="sinhala_kg_viz.html"):
    print(f"Loading RDF graph from {ttl_file}...")
    
    # 1. Load the RDF Graph
    g = rdflib.Graph()
    try:
        g.parse(ttl_file, format="turtle")
    except Exception as e:
        print(f"Error loading {ttl_file}: {e}")
        return

    # 2. Initialize PyVis Network (Interactive HTML canvas)
    # Using a dark theme which usually makes colored nodes pop better
    net = Network(height="800px", width="100%", bgcolor="#ffffff", font_color="black", directed=True)
    
    # Enable physics for auto-layout (spring layout)
    net.force_atlas_2based(gravity=-50)

    # Namespaces for easier parsing
    SIN = rdflib.Namespace("http://research.lk/ontology/sinhala#")
    RDF = rdflib.RDF
    RDFS = rdflib.RDFS

    print("Processing nodes and edges...")

    # 3. Helper function to get a clean label for a node
    def get_node_label(node):
        # If it's a literal, just return the text
        if isinstance(node, rdflib.Literal):
            return str(node)
        
        # If it has an rdfs:label in the graph, use that (e.g., "කා")
        labels = list(g.objects(node, RDFS.label))
        if labels:
            return str(labels[0])
            
        # Otherwise, extract the fragment from the URI (e.g., "Ka_AelaPilla")
        if isinstance(node, rdflib.URIRef):
            return node.split("#")[-1]
            
        return str(node)

    # 4. Helper function to determine node color based on its RDF Type
    def get_node_color(node):
        types = list(g.objects(node, RDF.type))
        if SIN.BaseConsonant in types:
            return "#6BAED6" # Light Blue
        elif SIN.Modifier in types:
            return "#74C476" # Green
        elif SIN.Syllable in types:
            return "#FB6A4A" # Red
        elif RDFS.Class in types:
            return "#9E9AC8" # Purple (For ontology classes)
        return "#CCCCCC"     # Gray for generic nodes/literals

    # 5. Add Nodes and Edges to the visualization
    added_nodes = set()

    for subj, pred, obj in g:
        # Skip purely definitional triples to keep the graph uncluttered
        # (e.g., skip 'hasBase is a Property')
        if pred == RDF.type and obj in [RDF.Property, RDFS.Class]:
            continue

        subj_id = str(subj)
        obj_id = str(obj)
        pred_label = str(pred).split("#")[-1] if "#" in str(pred) else str(pred)

        # Add Subject Node
        if subj_id not in added_nodes:
            net.add_node(subj_id, label=get_node_label(subj), color=get_node_color(subj), title=subj_id)
            added_nodes.add(subj_id)

        # Add Object Node (if it's not a literal string being used as an edge property)
        # For visualization, we often turn literal properties (like sounds/romanization) into node hover-text,
        # but here we'll plot them as actual nodes for completeness.
        if obj_id not in added_nodes:
            # If the object is a literal (like a string), make it a yellow sticky-note color
            color = "#FD8D3C" if isinstance(obj, rdflib.Literal) else get_node_color(obj)
            shape = "box" if isinstance(obj, rdflib.Literal) else "dot"
            
            net.add_node(obj_id, label=get_node_label(obj), color=color, shape=shape, title=obj_id)
            added_nodes.add(obj_id)

        # Add Edge
        net.add_edge(subj_id, obj_id, title=pred_label, label=pred_label)

    # 6. Generate and save the HTML file
    print(f"Generating visualization: {output_html}...")
    try:
        net.write_html(output_html)
        print("Done!")
        # Automatically open in the default web browser
        webbrowser.open('file://' + os.path.realpath(output_html))
    except Exception as e:
        print(f"Failed to write HTML: {e}")

if __name__ == "__main__":
    # WARNING: Visualizing the entire `sinhala_kg.ttl` (1000+ triples) might be slow and messy.
    # It is highly recommended to test this on a subgraph first!
    
    # 1. To visualize the whole alphabet (can be a hairball):
    # visualize_rdf_graph("sinhala_kg.ttl", "full_alphabet_viz.html")
    
    # 2. To visualize a specific word subgraph (Recommended):
    # First, ensure you ran the `extract_subgraph.py` script to save a small TTL file.
    # For example, if you saved the subgraph for "කාලය" to "subgraph_kaalaya.ttl":
    
    target_file = "sinhala_kg.ttl" # Change this to your specific subgraph file if you have one
    visualize_rdf_graph(target_file, "kg_visualization.html")import rdflib
import networkx as nx
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout # Optional, requires pygraphviz
import os

# --- Paste the full Tokenization and Extractor functions here for completion ---
def tokenize_sinhala_string(text):
    """Segmenter for Sinhala syllables."""
    tokens = []
    for char in text:
        is_modifier = ('\u0DCA' <= char <= '\u0DDF') or char in ['\u0D82', '\u0D83']
        if is_modifier:
            if tokens: tokens[-1] += char
            else: tokens.append(char)
        elif char.strip():
            tokens.append(char)
    return tokens

def extract_subgraph_with_viz(input_text, kg_path="sinhala_kg.ttl"):
    # 1. Initialize graphs
    main_graph = rdflib.Graph()
    if not os.path.exists(kg_path):
        print(f"Error: Foundational KG {kg_path} not found. Please ensure it's generated and uploaded.")
        return None
    main_graph.parse(kg_path, format="turtle")
    subgraph = rdflib.Graph()
    SIN = rdflib.Namespace("http://research.lk/ontology/sinhala#")
    subgraph.bind("sin", SIN)
    
    # 2. Extract subgraph (as before)
    tokens = tokenize_sinhala_string(input_text)
    for token in tokens:
        query = f"""
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX sin: <http://research.lk/ontology/sinhala#>
        CONSTRUCT {{
            ?node ?p ?o .
            ?base ?base_p ?base_o .
            ?mod ?mod_p ?mod_o .
        }}
        WHERE {{
            ?node rdfs:label "{token}"@si .
            ?node ?p ?o .
            OPTIONAL {{ ?node sin:hasBase ?base . ?base ?base_p ?base_o . }}
            OPTIONAL {{ ?node sin:modifiedBy ?mod . ?mod ?mod_p ?mod_o . }}
        }}
        """
        for triple in main_graph.query(query):
            subgraph.add(triple)
    return subgraph

def plot_static_subgraph(g, output_file="subgraph_viz.png"):
    if not g or len(g) == 0:
        print("Error: Graph is empty.")
        return

    # 1. Initialize NetworkX Graph and Namespaces
    G = nx.DiGraph()
    SIN = rdflib.Namespace("http://research.lk/ontology/sinhala#")
    RDF = rdflib.RDF
    RDFS = rdflib.RDFS

    # Define color scheme (as before: Blue=Base, Green=Modifier, Red=Syllable)
    COLORS = {SIN.BaseConsonant: "#6BAED6", SIN.Modifier: "#74C476", SIN.Syllable: "#FB6A4A"}

    # 2. Load nodes and edges from RDF to NetworkX
    node_colors = []
    labels = {}

    for subj, pred, obj in g:
        # Ignore literal properties like sounds/romanizations for clarity (just the graph structure)
        if isinstance(subj, rdflib.Literal) or isinstance(obj, rdflib.Literal):
            continue

        s_id = str(subj)
        o_id = str(obj)
        pred_label = str(pred).split("#")[-1]

        # Process Subject Node
        if s_id not in G:
            G.add_node(s_id)
            # Find its RDF type for coloring
            subj_type = list(g.objects(subj, RDF.type))[0] if list(g.objects(subj, RDF.type)) else None
            node_colors.append(COLORS.get(subj_type, "#9E9AC8")) # Use Purple for unknown/class nodes
            labels[s_id] = str(list(g.objects(subj, RDFS.label))[0]) if list(g.objects(subj, RDFS.label)) else s_id.split("#")[-1]

        # Process Object Node (unless it's an ontology class we are mapping *type* to)
        # If the predicate is `type`, the object is the header class. We just want to visualize the relationship.
        if pred == RDF.type:
            continue

        if o_id not in G:
            G.add_node(o_id)
            obj_type = list(g.objects(obj, RDF.type))[0] if list(g.objects(obj, RDF.type)) else None
            node_colors.append(COLORS.get(obj_type, "#9E9AC8"))
            labels[o_id] = str(list(g.objects(obj, RDFS.label))[0]) if list(g.objects(obj, RDFS.label)) else o_id.split("#")[-1]

        # Add Edge
        if pred != RDF.type:
             G.add_edge(s_id, o_id, label=pred_label)

    # 3. Setup Visualization Environment (similar aesthetics to watermarked diagram)
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.set_title("Sinhala Word Subgraph (Orthographic Rule Extraction)", color="#333333")

    # Layout: Try Graphviz if available (better), otherwise use spring_layout
    try:
        pos = graphviz_layout(G, prog='dot')
    except (ImportError, Exception):
        # Fallback to a clear vertical-ish layout using shell_layout or spring_layout
        pos = nx.spring_layout(G, k=1.0, iterations=100, seed=42)

    # 4. Draw Components
    # Draw Nodes (Circles/Dots as per standard graph, colored by type)
    nx.draw_networkx_nodes(G, pos, node_size=1200, node_color=node_colors, edgecolors='#333333', ax=ax, alpha=0.9)
    # Draw Labels inside Nodes (Sinhala text)
    nx.draw_networkx_labels(G, pos, labels, font_size=14, font_family='sans-serif', font_color='black', ax=ax)
    # Draw Directed Edges
    nx.draw_networkx_edges(G, pos, edgelist=G.edges(), edge_color="#666666", width=1.5, arrowstyle='->', arrowsize=15, connectionstyle='arc3,rad=0.1', ax=ax)
    # Draw Edge Labels (hasBase, modifiedBy)
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10, font_color='#666666', font_family='sans-serif', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9), ax=ax)

    # 5. Clean up the plot
    plt.axis('off')
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Test with "කාලය"
    word_input = "කාලය"
    # Word changed to show new structure visualization
    # word_input = "පාර" 

    print(f"Generating and Visualizing subgraph for input: '{word_input}'...")
    
    # 1. Perform extraction
    # Ensure sinhala_kg.ttl is present in /kaggle/working/ before running!
    current_subgraph = extract_subgraph_with_viz(word_input, "sinhala_kg.ttl")
    
    # 2. Visualize
    if current_subgraph and len(current_subgraph) > 0:
        plot_static_subgraph(current_subgraph)
    else:
        print("Subgraph extraction failed. Ensure your KG is correct.")

In [ ]:
import rdflib
import networkx as nx
import matplotlib.pyplot as plt
from networkx.drawing.nx_agraph import graphviz_layout # Optional, requires pygraphviz
import os

# --- Paste the full Tokenization and Extractor functions here for completion ---
def tokenize_sinhala_string(text):
    """Segmenter for Sinhala syllables."""
    tokens = []
    for char in text:
        is_modifier = ('\u0DCA' <= char <= '\u0DDF') or char in ['\u0D82', '\u0D83']
        if is_modifier:
            if tokens: tokens[-1] += char
            else: tokens.append(char)
        elif char.strip():
            tokens.append(char)
    return tokens

def extract_subgraph_with_viz(input_text, kg_path="sinhala_kg.ttl"):
    # 1. Initialize graphs
    main_graph = rdflib.Graph()
    if not os.path.exists(kg_path):
        print(f"Error: Foundational KG {kg_path} not found. Please ensure it's generated and uploaded.")
        return None
    main_graph.parse(kg_path, format="turtle")
    subgraph = rdflib.Graph()
    SIN = rdflib.Namespace("http://research.lk/ontology/sinhala#")
    subgraph.bind("sin", SIN)
    
    # 2. Extract subgraph (as before)
    tokens = tokenize_sinhala_string(input_text)
    for token in tokens:
        query = f"""
        PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
        PREFIX sin: <http://research.lk/ontology/sinhala#>
        CONSTRUCT {{
            ?node ?p ?o .
            ?base ?base_p ?base_o .
            ?mod ?mod_p ?mod_o .
        }}
        WHERE {{
            ?node rdfs:label "{token}"@si .
            ?node ?p ?o .
            OPTIONAL {{ ?node sin:hasBase ?base . ?base ?base_p ?base_o . }}
            OPTIONAL {{ ?node sin:modifiedBy ?mod . ?mod ?mod_p ?mod_o . }}
        }}
        """
        for triple in main_graph.query(query):
            subgraph.add(triple)
    return subgraph

def plot_static_subgraph(g, output_file="subgraph_viz.png"):
    if not g or len(g) == 0:
        print("Error: Graph is empty.")
        return

    # 1. Initialize NetworkX Graph and Namespaces
    G = nx.DiGraph()
    SIN = rdflib.Namespace("http://research.lk/ontology/sinhala#")
    RDF = rdflib.RDF
    RDFS = rdflib.RDFS

    # Define color scheme (as before: Blue=Base, Green=Modifier, Red=Syllable)
    COLORS = {SIN.BaseConsonant: "#6BAED6", SIN.Modifier: "#74C476", SIN.Syllable: "#FB6A4A"}

    # 2. Load nodes and edges from RDF to NetworkX
    node_colors = []
    labels = {}

    for subj, pred, obj in g:
        # Ignore literal properties like sounds/romanizations for clarity (just the graph structure)
        if isinstance(subj, rdflib.Literal) or isinstance(obj, rdflib.Literal):
            continue

        s_id = str(subj)
        o_id = str(obj)
        pred_label = str(pred).split("#")[-1]

        # Process Subject Node
        if s_id not in G:
            G.add_node(s_id)
            # Find its RDF type for coloring
            subj_type = list(g.objects(subj, RDF.type))[0] if list(g.objects(subj, RDF.type)) else None
            node_colors.append(COLORS.get(subj_type, "#9E9AC8")) # Use Purple for unknown/class nodes
            labels[s_id] = str(list(g.objects(subj, RDFS.label))[0]) if list(g.objects(subj, RDFS.label)) else s_id.split("#")[-1]

        # Process Object Node (unless it's an ontology class we are mapping *type* to)
        # If the predicate is `type`, the object is the header class. We just want to visualize the relationship.
        if pred == RDF.type:
            continue

        if o_id not in G:
            G.add_node(o_id)
            obj_type = list(g.objects(obj, RDF.type))[0] if list(g.objects(obj, RDF.type)) else None
            node_colors.append(COLORS.get(obj_type, "#9E9AC8"))
            labels[o_id] = str(list(g.objects(obj, RDFS.label))[0]) if list(g.objects(obj, RDFS.label)) else o_id.split("#")[-1]

        # Add Edge
        if pred != RDF.type:
             G.add_edge(s_id, o_id, label=pred_label)

    # 3. Setup Visualization Environment (similar aesthetics to watermarked diagram)
    fig, ax = plt.subplots(figsize=(10, 8))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.set_title("Sinhala Word Subgraph (Orthographic Rule Extraction)", color="#333333")

    # Layout: Try Graphviz if available (better), otherwise use spring_layout
    try:
        pos = graphviz_layout(G, prog='dot')
    except (ImportError, Exception):
        # Fallback to a clear vertical-ish layout using shell_layout or spring_layout
        pos = nx.spring_layout(G, k=1.0, iterations=100, seed=42)

    # 4. Draw Components
    # Draw Nodes (Circles/Dots as per standard graph, colored by type)
    nx.draw_networkx_nodes(G, pos, node_size=1200, node_color=node_colors, edgecolors='#333333', ax=ax, alpha=0.9)
    # Draw Labels inside Nodes (Sinhala text)
    nx.draw_networkx_labels(G, pos, labels, font_size=14, font_family='sans-serif', font_color='black', ax=ax)
    # Draw Directed Edges
    nx.draw_networkx_edges(G, pos, edgelist=G.edges(), edge_color="#666666", width=1.5, arrowstyle='->', arrowsize=15, connectionstyle='arc3,rad=0.1', ax=ax)
    # Draw Edge Labels (hasBase, modifiedBy)
    edge_labels = nx.get_edge_attributes(G, 'label')
    nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=10, font_color='#666666', font_family='sans-serif', bbox=dict(facecolor='white', edgecolor='none', alpha=0.9), ax=ax)

    # 5. Clean up the plot
    plt.axis('off')
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    # Test with "කාලය"
    # word_input = "කාලය"
    # Word changed to show new structure visualization
    word_input = "පාර" 

    print(f"Generating and Visualizing subgraph for input: '{word_input}'...")
    
    # 1. Perform extraction
    # Ensure sinhala_kg.ttl is present in /kaggle/working/ before running!
    current_subgraph = extract_subgraph_with_viz(word_input, "sinhala_kg.ttl")
    
    # 2. Visualize
    if current_subgraph and len(current_subgraph) > 0:
        plot_static_subgraph(current_subgraph)
    else:
        print("Subgraph extraction failed. Ensure your KG is correct.")

In [ ]:
import urllib.request
import matplotlib.font_manager as font_manager
import matplotlib.pyplot as plt
import os

font_url = "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/NotoSansSinhala/NotoSansSinhala-Regular.ttf"
font_path = "/kaggle/working/NotoSansSinhala.ttf"

if not os.path.exists(font_path):
    urllib.request.urlretrieve(font_url, font_path)

font_manager.fontManager.addfont(font_path)
sinhala_font = font_manager.FontProperties(fname=font_path)

# THE FIX: Pass a list. First try Sinhala, then try standard sans-serif for English
plt.rcParams['font.family'] = [sinhala_font.get_name(), 'sans-serif']

In [ ]:
# 1. Install necessary libraries
!pip install pyvis rdflib -q

import rdflib
from pyvis.network import Network
from IPython.display import HTML, display

def show_interactive_graph(g, output_file="subgraph_viz.html"):
    if not g or len(g) == 0:
        print("Graph is empty!")
        return

    # Initialize interactive network
    net = Network(height="600px", width="100%", bgcolor="#ffffff", font_color="black", directed=True, notebook=True)
    net.force_atlas_2based()
    
    # Add nodes and edges
    for subj, pred, obj in g:
        if isinstance(subj, rdflib.Literal) or isinstance(obj, rdflib.Literal):
            continue
            
        s_id, o_id = str(subj), str(obj)
        
        # Get labels
        s_label = str(list(g.objects(subj, rdflib.RDFS.label))[0]) if list(g.objects(subj, rdflib.RDFS.label)) else s_id.split("#")[-1]
        o_label = str(list(g.objects(obj, rdflib.RDFS.label))[0]) if list(g.objects(obj, rdflib.RDFS.label)) else o_id.split("#")[-1]
        pred_label = str(pred).split("#")[-1]

        # Map colors based on ontology types
        s_color = "#6BAED6" if "BaseConsonant" in s_id else "#FB6A4A" if "Syllable" in s_id else "#74C476"
        o_color = "#6BAED6" if "BaseConsonant" in o_id else "#FB6A4A" if "Syllable" in o_id else "#74C476"

        net.add_node(s_id, label=s_label, color=s_color)
        if pred != rdflib.RDF.type: # Ignore 'type' edges to keep the graph clean
            net.add_node(o_id, label=o_label, color=o_color)
            net.add_edge(s_id, o_id, title=pred_label, label=pred_label)

    # Save and display inside Kaggle
    net.write_html(output_file)
    display(HTML(filename=output_file))

# Assuming 'current_subgraph' is the graph you extracted in your previous cell
show_interactive_graph(current_subgraph)